In [0]:
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
import uuid

In [0]:
CATALOG = "retail"

In [0]:
def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()

In [0]:
def batch_id() -> str:
    return str(uuid.uuid4())

In [0]:
def log_run(layer: str, table: str, records: int, size_bytes: int, duration_s: float, status: str = "OK", error: str = None, batch: str = None):
    row = [(now_utc(), layer, table, records, size_bytes, round(duration_s, 2), status, error or "", batch or "")]
    schema = ("ts_log STRING, layer STRING, table_name STRING, records LONG, size_bytes LONG, duration_s DOUBLE, status STRING, error STRING, batch_id STRING")
    df = spark.createDataFrame(row, schema)
    (df.write.format("delta")
       .mode("append")
       .saveAsTable(f"{CATALOG}.bronze.pipeline_log"))

In [0]:
def log_referential_errors(df_errors: DataFrame, source_table: str, reason: str, batch: str):
    (df_errors.withColumn("ts_log", F.lit(now_utc()))
        .withColumn("source_table", F.lit(source_table))
        .withColumn("reason", F.lit(reason))
        .withColumn("batch_id", F.lit(batch))
        .write.format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOG}.silver.referential_errors"))

In [0]:
def upsert(df: DataFrame, target_table: str, pk_cols: list):
    """
    MERGE INTO target usando pk_cols como clave.
    Garantiza idempotencia en cualquier capa.
    """
    full_table = f"{CATALOG}.{target_table}"

    if not spark.catalog.tableExists(full_table):
        df.write.format("delta").mode("overwrite").saveAsTable(full_table)
        return

    target = DeltaTable.forName(spark, full_table)
    merge_cond = " AND ".join([f"target.{c} = source.{c}" for c in pk_cols])

    (target.alias("target")
        .merge(df.alias("source"), merge_cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())